In [ ]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / "README.md").exists()
            and (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise RuntimeError("Project root was not found. Open this notebook from inside the project folder.")

PROJECT_ROOT = find_project_root()
ROOT = PROJECT_ROOT
root = PROJECT_ROOT
project_root = PROJECT_ROOT

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root configured")
print("Working directory configured")


# Practical Growth Assessment Workflow

This notebook shows how the Feature Engineering and Time Series work can be used in a small practical workflow.

The goal is not to create a veterinary product. The goal is to demonstrate that the mathematical features can be applied to new Cane Corso measurements and turned into a readable educational report.


## Mathematical Idea

A repeated measurement can be represented as:

```text
r(t) = [age_months(t), weight_kg(t), height_cm(t)]
```

The practical workflow uses:

```text
weight_gain(t) = weight(t) - weight(t-1)
growth_velocity(t) = weight_gain(t) / delta_age(t)
z = (latest_velocity - reference_mean_velocity) / reference_standard_deviation
distance = sqrt(sum((x_latest_scaled - x_reference_scaled)^2))
```

This is a learning workflow: formulas first, then code, then interpretation.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Markdown, Image, display
project_root = next(candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / "src").is_dir() and (candidate / "data").is_dir())
if not (project_root / 'src').exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
project_root


## Load the Example Input

The input file behaves like a small owner record: one dog, repeated measurements over time.


In [ ]:
input_path = project_root / 'data' / 'input' / 'example_new_cane_corso_measurements.csv'
measurements = pd.read_csv(input_path)
measurements


## Run the Practical Workflow

The script engineers features, compares the latest record with the reference feature dataset, creates figures and writes a report.


In [ ]:
from src.run_growth_assessment import run_assessment

result = run_assessment()
assessment = result['assessment']

print('Practical signal:', assessment.practical_signal)
print('Latest velocity z-score:', round(assessment.velocity_z_score, 3))
print('Weight deviation percent:', round(assessment.weight_deviation_percent, 3))
print('Report:', result['report_path'])


## Engineered Output Features

The workflow creates a processed feature dataset that can be inspected like the previous course notebooks.


In [ ]:
features_path = project_root / 'data' / 'processed' / 'example_growth_assessment_features.csv'
features = pd.read_csv(features_path)
features.tail()


## Visual Output

The figures make the report easier to interpret.


In [ ]:
display(Image(filename=str(project_root / 'reports' / 'figures' / 'practical_growth_assessment_weight_trend.png')))


In [ ]:
display(Image(filename=str(project_root / 'reports' / 'figures' / 'practical_growth_assessment_velocity_signal.png')))


## Generated Report

The markdown report is the practical output. It combines formulas, values, interpretation and limitations.


In [ ]:
report_path = project_root / 'reports' / 'example_growth_assessment_report.md'
display(Markdown(report_path.read_text(encoding='utf-8')))


## Learning Reflection

This workflow is useful because it connects the course material to an applied use case. I am still learning the methods, so the report is intentionally careful and limited. It shows how mathematical features can organize measurements and generate monitoring signals, but it does not make medical or official claims.
